In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import netCDF4 as netcdf4
from scipy.interpolate import interp1d

path = "data_raw/"

### 2004

In [2]:
# create one nc file for every year
temp_RH_raw_2004 = xr.open_dataset(path +"2004_temp_RH_raw.nc", engine = "netcdf4")
temp_RH_raw_2004


<xarray.Dataset> Size: 1GB
Dimensions:         (valid_time: 13, pressure_level: 37, latitude: 121,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 104B 2004-01-04 ... 2004-01-16
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
  * latitude        (latitude) float64 968B 90.0 89.75 89.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    number          int64 8B ...
    expver          (valid_time) <U4 208B ...
Data variables:
    o3              (valid_time, pressure_level, latitude, longitude) float32 335MB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 335MB ...
    t               (valid_time, pressure_level, latitude, longitude) float32 335MB ...
    u               (valid_time, pressure_level, latitude, longitude) float32 335MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T12:36 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:
# extract relevant time
time = temp_RH_raw_2004.valid_time[1:-1]
pressure_temp_RH = temp_RH_raw_2004.pressure_level
rh = temp_RH_raw_2004.r[1:-1,:,:,:]
temp = temp_RH_raw_2004.t[1:-1,:,:,:]
u = temp_RH_raw_2004.u[1:-1,:,:,:]

# build mean over region (polar cap)
rh = np.mean(rh, axis = 3)
rh = np.mean(rh, axis = 2)
temp = np.mean(temp, axis = 3)
temp = np.mean(temp, axis = 2)
u = np.mean(u, axis = 3)
u = u[:,:, -1]  # take only 60 °N
print(u.shape)

(11, 37)


In [5]:
print(pressure_temp_RH.shape)
print(pressure_temp_RH)

(37,)
<xarray.DataArray 'pressure_level' (pressure_level: 37)> Size: 296B
array([1000.,  975.,  950.,  925.,  900.,  875.,  850.,  825.,  800.,  775.,
        750.,  700.,  650.,  600.,  550.,  500.,  450.,  400.,  350.,  300.,
        250.,  225.,  200.,  175.,  150.,  125.,  100.,   70.,   50.,   30.,
         20.,   10.,    7.,    5.,    3.,    2.,    1.])
Coordinates:
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
    number          int64 8B ...
Attributes:
    long_name:         pressure
    units:             hPa
    positive:          down
    stored_direction:  decreasing
    standard_name:     air_pressure


In [7]:
ds_co2_2004 = xr.open_dataset(path+"2004_CO2/2004_CO2.nc", engine = "netcdf4")
ds_co2_2004

<xarray.Dataset> Size: 22MB
Dimensions:         (valid_time: 11, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 88B 2004-01-05 ... 2004-01-15
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    co2             (valid_time, pressure_level, latitude, longitude) float32 22MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T13:42 GRIB to CDM+CF via cfgrib-0.9.1...

In [8]:
co2_2004 = ds_co2_2004.co2
pressure_chemicals = ds_co2_2004.pressure_level

# mean over area
co2_2004 = np.mean(co2_2004, axis = 3)
co2_2004 = np.mean(co2_2004, axis = 2)
print(co2_2004.shape)

(11, 25)


In [13]:
def interp_pressure(data, p_src, p_tgt, axis=2):
    f = interp1d(
        p_src,
        data,
        axis=axis,
        kind="linear",
        bounds_error=False,
        fill_value=np.nan
    )
    return f(p_tgt)

In [14]:
co2_2004 = interp_pressure(co2_2004, pressure_chemicals, pressure_temp_RH)
print(co2_2004.shape)

(11, 37)


In [16]:
ds_nox_o3 = xr.open_dataset(path+"2004_NOX_O3/2004_NOX_O3.nc", engine = "netcdf4")
ds_nox_o3

<xarray.Dataset> Size: 65MB
Dimensions:         (valid_time: 11, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 88B 2004-01-05 ... 2004-01-15
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no2             (valid_time, pressure_level, latitude, longitude) float32 22MB ...
    no              (valid_time, pressure_level, latitude, longitude) float32 22MB ...
    go3             (valid_time, pressure_level, latitude, longitude) float32 22MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:51 GRIB to CDM+CF via cfgrib-0.9.1...

In [17]:
no2 = ds_nox_o3.no2
no = ds_nox_o3.no
o3 = ds_nox_o3.go3

# mean over area:
no2 = np.mean(no2, axis = 3)
no2 = np.mean(no2, axis = 2)
no2 = interp_pressure(no2, pressure_chemicals, pressure_temp_RH)

no = np.mean(no, axis = 3)
no = np.mean(no, axis = 2)
no = interp_pressure(no, pressure_chemicals, pressure_temp_RH)

o3 = np.mean(o3, axis = 3)
o3 = np.mean(o3, axis = 2)
o3 = interp_pressure(o3, pressure_chemicals, pressure_temp_RH)

In [18]:
data_2004 = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_temp_RH"), co2_2004.data),
        "no2": (("time", "pressure_temp_RH"), no2.data),
        "no": (("time", "pressure_temp_RH"), no.data),
        "o3": (("time", "pressure_temp_RH"), o3.data),
        "t": (("time", "pressure_temp_RH"), temp.data),
        "u": (("time", "pressure_temp_RH"), u.data),
        "rh": (("time", "pressure_temp_RH"), rh.data),

    },
    coords={
        "time": time,
        "pressure": pressure_temp_RH,
    },
)

data_2004.to_netcdf("data/2004_data.nc", engine = "netcdf4")

In [14]:
# create new nc-file:
ds_2004_chemicals = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_chemicals"), co2_2004.data),
        "no2": (("time", "pressure_chemicals"), no2.data),
        "no": (("time", "pressure_chemicals"), no.data),
        "o3": (("time", "pressure_chemicals"), o3.data),
    },
    coords={
        "time": time,
        "pressure_chemicals": pressure_chemicals,
    },
)

ds_2004_chemicals.to_netcdf("data/2004_chemicals.nc", engine = "netcdf4")


ds_2004 = xr.Dataset(
    data_vars={
        "t" : (("time", "pressure_temp_RH"), temp.data),
        "rh": (("time", "pressure_temp_RH"), rh.data),
        "u": (("time", "pressure_temp_RH"), u.data),
    },
    coords={
        "time": time,
        "pressure_temp_RH": pressure_temp_RH,
    }
)

ds_2004.to_netcdf("data/2004_temp_RH_u.nc", engine = "netcdf4")

### 2006

In [20]:
temp_RH_raw_2006 = xr.open_dataset(path +"2006_temp_RH_raw.nc", engine = "netcdf4")
temp_RH_raw_2006

<xarray.Dataset> Size: 6GB
Dimensions:         (valid_time: 59, pressure_level: 37, latitude: 121,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 472B 2006-01-01 ... 2006-02-28
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
  * latitude        (latitude) float64 968B 90.0 89.75 89.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    number          int64 8B ...
    expver          (valid_time) <U4 944B ...
Data variables:
    o3              (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    t               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    u               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T12:52 GRIB to CDM+CF via cfgrib-0.9.1...

In [21]:

# extract relevant time
time_2006 = temp_RH_raw_2006.valid_time[1:-1]
time_2006 = time_2006[19:46]

In [22]:
# extract relevant time

pressure_temp_RH_2006 = temp_RH_raw_2006.pressure_level
rh_2006 = temp_RH_raw_2006.r[19:46, :, :, :]
temp_2006 = temp_RH_raw_2006.t[19:46, :, :, :]
u_2006 = temp_RH_raw_2006.u[19:46, :, :, :]

# build mean over region (polar cap)
rh_2006 = np.mean(rh_2006, axis=3)
rh_2006 = np.mean(rh_2006, axis=2)
temp_2006 = np.mean(temp_2006, axis=3)
temp_2006 = np.mean(temp_2006, axis=2)
u_2006 = np.mean(u_2006, axis=3)
u_2006 = u_2006[:, :, -1]  # take only 60 °N
print(u_2006.shape)


ds_2006 = xr.Dataset(
    data_vars={
        "t" : (("time", "pressure_temp_RH"), temp_2006.data),
        "rh": (("time", "pressure_temp_RH"), rh_2006.data),
        "u": (("time", "pressure_temp_RH"), u_2006.data),
    },
    coords={
        "time": time_2006,
        "pressure_temp_RH": pressure_temp_RH_2006,
    }
)

ds_2006.to_netcdf("data/2006_temp_RH_u.nc", engine = "netcdf4")

(27, 37)


In [23]:
ds_co2_2006 = xr.open_dataset(path+"2006_CO2/2006_CO2.nc", engine = "netcdf4")
ds_co2_2006

<xarray.Dataset> Size: 53MB
Dimensions:         (valid_time: 27, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 216B 2006-01-21 ... 2006-02-16
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    co2             (valid_time, pressure_level, latitude, longitude) float32 53MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T16:36 GRIB to CDM+CF via cfgrib-0.9.1...

In [26]:

co2_2006 = ds_co2_2006.co2
pressure_chemicals_2006 = ds_co2_2006.pressure_level

# mean over area
co2_2006 = np.mean(co2_2006, axis=3)
co2_2006 = np.mean(co2_2006, axis=2)
print(co2_2006.shape)
co2_2006 = interp_pressure(co2_2006, pressure_chemicals_2006, pressure_temp_RH_2006)
print(co2_2006.shape)

(27, 25)
(27, 37)


In [27]:
ds_no2_2006 = xr.open_dataset(path+"2006_NO2/2006_NO2.nc", engine = "netcdf4")
ds_no2_2006

<xarray.Dataset> Size: 53MB
Dimensions:         (valid_time: 27, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 216B 2006-01-21 ... 2006-02-16
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no2             (valid_time, pressure_level, latitude, longitude) float32 53MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:54 GRIB to CDM+CF via cfgrib-0.9.1...

In [28]:
no2_2006 = ds_no2_2006.no2


# mean over area:
no2_2006 = np.mean(no2_2006, axis = 3)
no2_2006 = np.mean(no2_2006, axis = 2)
no2_2006 = interp_pressure(no2_2006, pressure_chemicals_2006, pressure_temp_RH_2006)

In [29]:
ds_no_o3_2006 = xr.open_dataset(path+"2006_NO_O3/2006_NO_O3.nc", engine = "netcdf4")
ds_no_o3_2006

<xarray.Dataset> Size: 106MB
Dimensions:         (valid_time: 27, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 216B 2006-01-21 ... 2006-02-16
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no              (valid_time, pressure_level, latitude, longitude) float32 53MB ...
    go3             (valid_time, pressure_level, latitude, longitude) float32 53MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:53 GRIB to CDM+CF via cfgrib-0.9.1...

In [30]:
no_2006 = ds_no_o3_2006.no
o3_2006 = ds_no_o3_2006.go3

no_2006 = np.mean(no_2006, axis = 3)
no_2006 = np.mean(no_2006, axis = 2)
no_2006 = interp_pressure(no_2006, pressure_chemicals_2006, pressure_temp_RH_2006)

o3_2006 = np.mean(o3_2006, axis = 3)
o3_2006 = np.mean(o3_2006, axis = 2)
o3_2006 = interp_pressure(o3_2006, pressure_chemicals_2006, pressure_temp_RH_2006)

In [31]:
data_2006 = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_temp_RH"), co2_2006.data),
        "no2": (("time", "pressure_temp_RH"), no2_2006.data),
        "no": (("time", "pressure_temp_RH"), no_2006.data),
        "o3": (("time", "pressure_temp_RH"), o3_2006.data),
        "t" : (("time", "pressure_temp_RH"), temp_2006.data),
        "u": (("time", "pressure_temp_RH"), u_2006.data),
        "rh" : (("time", "pressure_temp_RH"), rh_2006.data),
    },
    coords={
        "time": time_2006,
        "pressure": pressure_temp_RH_2006,
    },
)

data_2006.to_netcdf("data/2006_data.nc", engine = "netcdf4")

### 2007

In [32]:
temp_RH_raw_2007 = xr.open_dataset(path +"2007_temp_RH_raw.nc", engine = "netcdf4")
temp_RH_raw_2007

<xarray.Dataset> Size: 2GB
Dimensions:         (valid_time: 19, pressure_level: 37, latitude: 121,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 152B 2007-02-01 ... 2007-03-31
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
  * latitude        (latitude) float64 968B 90.0 89.75 89.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    number          int64 8B ...
    expver          (valid_time) <U4 304B ...
Data variables:
    o3              (valid_time, pressure_level, latitude, longitude) float32 490MB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 490MB ...
    t               (valid_time, pressure_level, latitude, longitude) float32 490MB ...
    u               (valid_time, pressure_level, latitude, longitude) float32 490MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T13:09 GRIB to CDM+CF via cfgrib-0.9.1...

In [33]:
time_2007 = temp_RH_raw_2007.valid_time
time_2007 = time_2007[3:9]

pressure_temp_RH_2007 = temp_RH_raw_2007.pressure_level
rh_2007 = temp_RH_raw_2007.r[3:9, :, :, :]
temp_2007 = temp_RH_raw_2007.t[3:9, :, :, :]
u_2007 = temp_RH_raw_2007.u[3:9, :, :, :]

# build mean over region (polar cap)
rh_2007 = np.mean(rh_2007, axis=3)
rh_2007 = np.mean(rh_2007, axis=2)
temp_2007 = np.mean(temp_2007, axis=3)
temp_2007 = np.mean(temp_2007, axis=2)
u_2007 = np.mean(u_2007, axis=3)
u_2007 = u_2007[:, :, -1]  # take only 60 °N
print(u_2007.shape)

ds_2007 = xr.Dataset(
    data_vars={
        "t" : (("time", "pressure_temp_RH"), temp_2007.data),
        "rh": (("time", "pressure_temp_RH"), rh_2007.data),
        "u": (("time", "pressure_temp_RH"), u_2007.data),
    },
    coords={
        "time": time_2007,
        "pressure_temp_RH": pressure_temp_RH_2007,
    }
)

ds_2007.to_netcdf("data/2007_temp_RH_u.nc", engine = "netcdf4")

(6, 37)


In [34]:
ds_co2_2007 = xr.open_dataset(path+"2007_CO2/2007_CO2.nc", engine = "netcdf4")
ds_co2_2007

<xarray.Dataset> Size: 12MB
Dimensions:         (valid_time: 6, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 48B 2007-02-24 ... 2007-03-01
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    co2             (valid_time, pressure_level, latitude, longitude) float32 12MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:49 GRIB to CDM+CF via cfgrib-0.9.1...

In [36]:
pressure_chemicals_2007 = ds_co2_2007.pressure_level

co2_2007 = ds_co2_2007.co2
co2_2007 = np.mean(co2_2007, axis=3)
co2_2007 = np.mean(co2_2007, axis=2)
co2_2007 = interp_pressure(co2_2007, pressure_chemicals_2007, pressure_temp_RH_2007)



In [37]:
ds_nox_o3_2007 = xr.open_dataset(path+"2007_NOX_O3/2007_NOX_O3.nc", engine = "netcdf4")
ds_nox_o3_2007

<xarray.Dataset> Size: 35MB
Dimensions:         (valid_time: 6, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 48B 2007-02-24 ... 2007-03-01
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no2             (valid_time, pressure_level, latitude, longitude) float32 12MB ...
    no              (valid_time, pressure_level, latitude, longitude) float32 12MB ...
    go3             (valid_time, pressure_level, latitude, longitude) float32 12MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:55 GRIB to CDM+CF via cfgrib-0.9.1...

In [38]:
no2_2007 = ds_nox_o3_2007.no2
no_2007 = ds_nox_o3_2007.no
o3_2007 = ds_nox_o3_2007.go3

# mean over area:
no2_2007 = np.mean(no2_2007, axis = 3)
no2_2007 = np.mean(no2_2007, axis = 2)
no2_2007 = interp_pressure(no2_2007, pressure_chemicals_2007, pressure_temp_RH_2007)

no_2007 = np.mean(no_2007, axis = 3)
no_2007 = np.mean(no_2007, axis = 2)
no_2007 = interp_pressure(no_2007, pressure_chemicals_2007, pressure_temp_RH_2007)

o3_2007 = np.mean(o3_2007, axis = 3)
o3_2007 = np.mean(o3_2007, axis = 2)
o3_2007 = interp_pressure(o3_2007, pressure_chemicals_2007, pressure_temp_RH_2007)

In [39]:
# create new nc-file:
data_2007 = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_temp_RH"), co2_2007.data),
        "no2": (("time", "pressure_temp_RH"), no2_2007.data),
        "no": (("time", "pressure_temp_RH"), no_2007.data),
        "o3": (("time", "pressure_temp_RH"), o3_2007.data),
        "t" : (("time", "pressure_temp_RH"), temp_2007.data),
        "u" : (("time", "pressure_temp_RH"), u_2007.data),
        "rh" : (("time", "pressure_temp_RH"), rh_2007.data),
    },
    coords={
        "time": time_2007,
        "pressure": pressure_temp_RH_2007,
    },
)

data_2007.to_netcdf("data/2007_data.nc", engine = "netcdf4")

### 2008

In [40]:
temp_RH_raw_2008 = xr.open_dataset(path +"2008_temp_RH_raw.nc", engine = "netcdf4")
temp_RH_raw_2008

<xarray.Dataset> Size: 6GB
Dimensions:         (valid_time: 60, pressure_level: 37, latitude: 121,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 480B 2008-02-01 ... 2008-03-31
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
  * latitude        (latitude) float64 968B 90.0 89.75 89.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    number          int64 8B ...
    expver          (valid_time) <U4 960B ...
Data variables:
    o3              (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    t               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    u               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T13:25 GRIB to CDM+CF via cfgrib-0.9.1...

In [61]:
time_2008 = temp_RH_raw_2008.valid_time
time_2008 = time_2008[21:38]
print(time_2008)

pressure_temp_RH_2008 = temp_RH_raw_2008.pressure_level
rh_2008 = temp_RH_raw_2008.r[21:38, :, :, :]
temp_2008 = temp_RH_raw_2008.t[21:38, :, :, :]
u_2008 = temp_RH_raw_2008.u[21:38, :, :, :]

# build mean over region (polar cap)
rh_2008 = np.mean(rh_2008, axis=3)
rh_2008 = np.mean(rh_2008, axis=2)
temp_2008 = np.mean(temp_2008, axis=3)
temp_2008 = np.mean(temp_2008, axis=2)
u_2008 = np.mean(u_2008, axis=3)
u_2008 = u_2008[:, :, -1]  # take only 60 °N
print(u_2008.shape)

ds_2008 = xr.Dataset(
    data_vars={
        "t" : (("time", "pressure_temp_RH"), temp_2008.data),
        "rh": (("time", "pressure_temp_RH"), rh_2008.data),
        "u": (("time", "pressure_temp_RH"), u_2008.data),
    },
    coords={
        "time": time_2008,
        "pressure_temp_RH": pressure_temp_RH_2008,
    }
)

ds_2008.to_netcdf("data/2008_temp_RH_u.nc", engine = "netcdf4")

<xarray.DataArray 'valid_time' (valid_time: 17)> Size: 136B
array(['2008-02-22T00:00:00.000000000', '2008-02-23T00:00:00.000000000',
       '2008-02-24T00:00:00.000000000', '2008-02-25T00:00:00.000000000',
       '2008-02-26T00:00:00.000000000', '2008-02-27T00:00:00.000000000',
       '2008-02-28T00:00:00.000000000', '2008-02-29T00:00:00.000000000',
       '2008-03-01T00:00:00.000000000', '2008-03-02T00:00:00.000000000',
       '2008-03-03T00:00:00.000000000', '2008-03-04T00:00:00.000000000',
       '2008-03-05T00:00:00.000000000', '2008-03-06T00:00:00.000000000',
       '2008-03-07T00:00:00.000000000', '2008-03-08T00:00:00.000000000',
       '2008-03-09T00:00:00.000000000'], dtype='datetime64[ns]')
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 136B 2008-02-22 ... 2008-03-09
    number      int64 8B 0
    expver      (valid_time) <U4 272B ...
Attributes:
    long_name:      time
    standard_name:  time
(17, 37)


In [62]:
ds_co2_2008 = xr.open_dataset(path+"2008_CO2/2008_CO2.nc", engine = "netcdf4")
ds_co2_2008

<xarray.Dataset> Size: 33MB
Dimensions:         (valid_time: 17, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 136B 2008-02-22 ... 2008-03-09
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    co2             (valid_time, pressure_level, latitude, longitude) float32 33MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T16:08 GRIB to CDM+CF via cfgrib-0.9.1...

In [63]:
pressure_chemicals_2008 = ds_co2_2008.pressure_level

co2_2008 = ds_co2_2008.co2
co2_2008 = np.mean(co2_2008, axis=3)
co2_2008 = np.mean(co2_2008, axis=2)
co2_2008 = interp_pressure(co2_2008, pressure_chemicals_2008, pressure_temp_RH_2008)


In [64]:
ds_nox_o3_2008 = xr.open_dataset(path+"2008_NOX_O3/2008_NOX_O3.nc", engine = "netcdf4")
ds_nox_o3_2008

<xarray.Dataset> Size: 100MB
Dimensions:         (valid_time: 17, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 136B 2008-02-22 ... 2008-03-09
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no2             (valid_time, pressure_level, latitude, longitude) float32 33MB ...
    no              (valid_time, pressure_level, latitude, longitude) float32 33MB ...
    go3             (valid_time, pressure_level, latitude, longitude) float32 33MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:59 GRIB to CDM+CF via cfgrib-0.9.1...

In [65]:
no2_2008 = ds_nox_o3_2008.no2
no_2008 = ds_nox_o3_2008.no
o3_2008 = ds_nox_o3_2008.go3

# mean over area:
no2_2008 = np.mean(no2_2008, axis = 3)
no2_2008 = np.mean(no2_2008, axis = 2)
no2_2008 = interp_pressure(no2_2008, pressure_chemicals_2008, pressure_temp_RH_2008)

no_2008 = np.mean(no_2008, axis = 3)
no_2008 = np.mean(no_2008, axis = 2)
no_2008 = interp_pressure(no_2008, pressure_chemicals_2008, pressure_temp_RH_2008)

o3_2008 = np.mean(o3_2008, axis = 3)
o3_2008 = np.mean(o3_2008, axis = 2)
o3_2008 = interp_pressure(o3_2008, pressure_chemicals_2008, pressure_temp_RH_2008)

In [67]:
# create new nc-file:
data_2008 = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_temp_RH"), co2_2008.data),
        "no2": (("time", "pressure_temp_RH"), no2_2008.data),
        "no": (("time", "pressure_temp_RH"), no_2008.data),
        "o3": (("time", "pressure_temp_RH"), o3_2008.data),
        "t" : (("time", "pressure_temp_RH"), temp_2008.data),
        "u" : (("time", "pressure_temp_RH"), u_2008.data),
        "rh" : (("time", "pressure_temp_RH"), rh_2008.data),
    },
    coords={
        "time": time_2008,
        "pressure": pressure_temp_RH_2008,
    },
)

data_2008.to_netcdf("data/2008_data.nc", engine = "netcdf4")

### 2009

In [68]:
temp_RH_raw_2009 = xr.open_dataset(path +"2009_temp_RH_raw.nc", engine = "netcdf4")
temp_RH_raw_2009

<xarray.Dataset> Size: 6GB
Dimensions:         (valid_time: 59, pressure_level: 37, latitude: 121,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 472B 2009-01-01 ... 2009-02-28
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
  * latitude        (latitude) float64 968B 90.0 89.75 89.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    number          int64 8B ...
    expver          (valid_time) <U4 944B ...
Data variables:
    o3              (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    t               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
    u               (valid_time, pressure_level, latitude, longitude) float32 2GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T13:47 GRIB to CDM+CF via cfgrib-0.9.1...

In [69]:
time_2009 = temp_RH_raw_2009.valid_time
time_2009 = time_2009[23:54]

pressure_temp_RH_2009 = temp_RH_raw_2009.pressure_level
rh_2009 = temp_RH_raw_2009.r[23:54, :, :, :]
temp_2009 = temp_RH_raw_2009.t[23:54, :, :, :]
u_2009 = temp_RH_raw_2009.u[23:54, :, :, :]

# build mean over region (polar cap)
rh_2009 = np.mean(rh_2009, axis=3)
rh_2009 = np.mean(rh_2009, axis=2)
temp_2009 = np.mean(temp_2009, axis=3)
temp_2009 = np.mean(temp_2009, axis=2)
u_2009 = np.mean(u_2009, axis=3)
u_2009 = u_2009[:, :, -1]  # take only 60 °N
print(u_2009.shape)

ds_2009 = xr.Dataset(
    data_vars={
        "t" : (("time", "pressure_temp_RH"), temp_2009.data),
        "rh": (("time", "pressure_temp_RH"), rh_2009.data),
        "u": (("time", "pressure_temp_RH"), u_2009.data),
    },
    coords={
        "time": time_2009,
        "pressure_temp_RH": pressure_temp_RH_2009,
    }
)

ds_2009.to_netcdf("data/2009_temp_RH_u.nc", engine = "netcdf4")

(31, 37)


In [70]:
ds_co2_2009 = xr.open_dataset(path+"2009_CO2/2009_CO2.nc", engine = "netcdf4")
ds_co2_2009

<xarray.Dataset> Size: 61MB
Dimensions:         (valid_time: 31, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 248B 2009-01-24 ... 2009-02-23
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    co2             (valid_time, pressure_level, latitude, longitude) float32 61MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T16:44 GRIB to CDM+CF via cfgrib-0.9.1...

In [71]:
co2_2009 = ds_co2_2009.co2
pressure_chemicals_2009 = ds_co2_2009.pressure_level

# mean over area
co2_2009 = np.mean(co2_2009, axis=3)
co2_2009 = np.mean(co2_2009, axis=2)
print(co2_2009.shape)
co2_2009 = interp_pressure(co2_2009, pressure_chemicals_2009, pressure_temp_RH_2009)

(31, 25)


In [72]:
ds_no2_2009 = xr.open_dataset(path+"2009_NO2/2009_NO2.nc", engine = "netcdf4")
ds_no2_2009

<xarray.Dataset> Size: 61MB
Dimensions:         (valid_time: 31, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 248B 2009-01-24 ... 2009-02-23
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no2             (valid_time, pressure_level, latitude, longitude) float32 61MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T15:02 GRIB to CDM+CF via cfgrib-0.9.1...

In [73]:
no2_2009 = ds_no2_2009.no2


# mean over area:
no2_2009 = np.mean(no2_2009, axis = 3)
no2_2009 = np.mean(no2_2009, axis = 2)
no2_2009 = interp_pressure(no2_2009, pressure_chemicals_2009, pressure_temp_RH_2009)

In [74]:
ds_no_o3_2009 = xr.open_dataset(path+"2009_NO_O3/data_plev.nc", engine = "netcdf4")
ds_no_o3_2009

<xarray.Dataset> Size: 122MB
Dimensions:         (valid_time: 31, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 248B 2009-01-24 ... 2009-02-23
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no              (valid_time, pressure_level, latitude, longitude) float32 61MB ...
    go3             (valid_time, pressure_level, latitude, longitude) float32 61MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T15:02 GRIB to CDM+CF via cfgrib-0.9.1...

In [75]:

no_2009 = ds_no_o3_2009.no
o3_2009 = ds_no_o3_2009.go3

no_2009 = np.mean(no_2009, axis=3)
no_2009 = np.mean(no_2009, axis=2)
no_2009 = interp_pressure(no_2009, pressure_chemicals_2009, pressure_temp_RH_2009)

o3_2009 = np.mean(o3_2009, axis=3)
o3_2009 = np.mean(o3_2009, axis=2)
o3_2009 = interp_pressure(o3_2009, pressure_chemicals_2009, pressure_temp_RH_2009)

In [76]:
data_2009 = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_temp_RH"), co2_2009.data),
        "no2": (("time", "pressure_temp_RH"), no2_2009.data),
        "no": (("time", "pressure_temp_RH"), no_2009.data),
        "o3": (("time", "pressure_temp_RH"), o3_2009.data),
        "t" : (("time", "pressure_temp_RH"), temp_2009.data),
        "u" : (("time", "pressure_temp_RH"), u_2009.data),
        "rh": (("time", "pressure_temp_RH"), rh_2009.data),
    },
    coords={
        "time": time_2009,
        "pressure": pressure_temp_RH_2009,
    },
)

data_2009.to_netcdf("data/2009_data.nc", engine = "netcdf4")

### 2010

In [77]:
temp_RH_raw_2010 = xr.open_dataset(path +"2010_temp_RH_raw.nc", engine = "netcdf4")
temp_RH_raw_2010

<xarray.Dataset> Size: 3GB
Dimensions:         (valid_time: 28, pressure_level: 37, latitude: 121,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 224B 2010-02-01 ... 2010-02-28
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
  * latitude        (latitude) float64 968B 90.0 89.75 89.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    number          int64 8B ...
    expver          (valid_time) <U4 448B ...
Data variables:
    o3              (valid_time, pressure_level, latitude, longitude) float32 722MB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 722MB ...
    t               (valid_time, pressure_level, latitude, longitude) float32 722MB ...
    u               (valid_time, pressure_level, latitude, longitude) float32 722MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:02 GRIB to CDM+CF via cfgrib-0.9.1...

In [78]:
time_2010 = temp_RH_raw_2010.valid_time
time_2010 = time_2010[8:19]

pressure_temp_RH_2010 = temp_RH_raw_2010.pressure_level
rh_2010 = temp_RH_raw_2010.r[8:19, :, :, :]
temp_2010 = temp_RH_raw_2010.t[8:19, :, :, :]
u_2010 = temp_RH_raw_2010.u[8:19, :, :, :]

# build mean over region (polar cap)
rh_2010 = np.mean(rh_2010, axis=3)
rh_2010 = np.mean(rh_2010, axis=2)
temp_2010 = np.mean(temp_2010, axis=3)
temp_2010 = np.mean(temp_2010, axis=2)
u_2010 = np.mean(u_2010, axis=3)
u_2010 = u_2010[:, :, -1]  # take only 60 °N
print(u_2010.shape)

ds_2010 = xr.Dataset(
    data_vars={
        "t" : (("time", "pressure_temp_RH"), temp_2010.data),
        "rh": (("time", "pressure_temp_RH"), rh_2010.data),
        "u": (("time", "pressure_temp_RH"), u_2010.data),
    },
    coords={
        "time": time_2010,
        "pressure_temp_RH": pressure_temp_RH_2010,
    }
)

ds_2010.to_netcdf("data/2010_temp_RH_u.nc", engine = "netcdf4")

(11, 37)


In [79]:
ds_co2_2010 = xr.open_dataset(path+"2010_CO2/2010_CO2.nc", engine = "netcdf4")
ds_co2_2010

<xarray.Dataset> Size: 22MB
Dimensions:         (valid_time: 11, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 88B 2010-02-09 ... 2010-02-19
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    co2             (valid_time, pressure_level, latitude, longitude) float32 22MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:06 GRIB to CDM+CF via cfgrib-0.9.1...

In [80]:
co2_2010 = ds_co2_2010.co2
pressure_chemicals_2010 = ds_co2_2010.pressure_level

# mean over area
co2_2010 = np.mean(co2_2010, axis=3)
co2_2010 = np.mean(co2_2010, axis=2)
print(co2_2010.shape)
co2_2010 = interp_pressure(co2_2010, pressure_chemicals_2010, pressure_temp_RH_2010)

(11, 25)


In [81]:
ds_nox_o3_2010 = xr.open_dataset(path+"2010_NOX_O3/2010_NOX_O3.nc", engine = "netcdf4")
ds_nox_o3_2010

<xarray.Dataset> Size: 65MB
Dimensions:         (valid_time: 11, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 88B 2010-02-09 ... 2010-02-19
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no2             (valid_time, pressure_level, latitude, longitude) float32 22MB ...
    no              (valid_time, pressure_level, latitude, longitude) float32 22MB ...
    go3             (valid_time, pressure_level, latitude, longitude) float32 22MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:57 GRIB to CDM+CF via cfgrib-0.9.1...

In [82]:
no2_2010 = ds_nox_o3_2010.no2
no_2010 = ds_nox_o3_2010.no
o3_2010 = ds_nox_o3_2010.go3

# mean over area:
no2_2010 = np.mean(no2_2010, axis = 3)
no2_2010 = np.mean(no2_2010, axis = 2)
no2_2010 = interp_pressure(no2_2010, pressure_chemicals_2010, pressure_temp_RH_2010)

no_2010 = np.mean(no_2010, axis = 3)
no_2010 = np.mean(no_2010, axis = 2)
no_2010 = interp_pressure(no_2010, pressure_chemicals_2010, pressure_temp_RH_2010)

o3_2010 = np.mean(o3_2010, axis = 3)
o3_2010 = np.mean(o3_2010, axis = 2)
o3_2010 = interp_pressure(o3_2010, pressure_chemicals_2010, pressure_temp_RH_2010)

In [83]:
# create new nc-file:
data_2010 = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_temp_RH"), co2_2010.data),
        "no2": (("time", "pressure_temp_RH"), no2_2010.data),
        "no": (("time", "pressure_temp_RH"), no_2010.data),
        "o3": (("time", "pressure_temp_RH"), o3_2010.data),
        "t": (("time", "pressure_temp_RH"), temp_2010.data),
        "u": (("time", "pressure_temp_RH"), u_2010.data),
        "rh": (("time", "pressure_temp_RH"), rh_2010.data),
    },
    coords={
        "time": time_2010,
        "pressure": pressure_temp_RH_2010,
    },
)

data_2010.to_netcdf("data/2010_data.nc", engine = "netcdf4")

### 2013

In [84]:
temp_RH_raw_2013 = xr.open_dataset(path +"2013_temp_RH_raw.nc", engine = "netcdf4")
temp_RH_raw_2013

<xarray.Dataset> Size: 3GB
Dimensions:         (valid_time: 31, pressure_level: 37, latitude: 121,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 248B 2013-01-01 ... 2013-01-31
  * pressure_level  (pressure_level) float64 296B 1e+03 975.0 950.0 ... 2.0 1.0
  * latitude        (latitude) float64 968B 90.0 89.75 89.5 ... 60.5 60.25 60.0
  * longitude       (longitude) float64 12kB -180.0 -179.8 ... 179.5 179.8
    number          int64 8B ...
    expver          (valid_time) <U4 496B ...
Data variables:
    o3              (valid_time, pressure_level, latitude, longitude) float32 799MB ...
    r               (valid_time, pressure_level, latitude, longitude) float32 799MB ...
    t               (valid_time, pressure_level, latitude, longitude) float32 799MB ...
    u               (valid_time, pressure_level, latitude, longitude) float32 799MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:28 GRIB to CDM+CF via cfgrib-0.9.1...

In [91]:
time_2013 = temp_RH_raw_2013.valid_time
time_2013 = time_2013[6:27]


pressure_temp_RH_2013 = temp_RH_raw_2013.pressure_level
rh_2013 = temp_RH_raw_2013.r[6:27, :, :, :]
temp_2013 = temp_RH_raw_2013.t[6:27, :, :, :]
u_2013 = temp_RH_raw_2013.u[6:27, :, :, :]

# build mean over region (polar cap)
rh_2013 = np.mean(rh_2013, axis=3)
rh_2013 = np.mean(rh_2013, axis=2)
temp_2013 = np.mean(temp_2013, axis=3)
temp_2013 = np.mean(temp_2013, axis=2)
u_2013 = np.mean(u_2013, axis=3)
u_2013 = u_2013[:, :, -1]  # take only 60 °N
print(u_2013.shape)

ds_2013 = xr.Dataset(
    data_vars={
        "t" : (("time", "pressure_temp_RH"), temp_2013.data),
        "rh": (("time", "pressure_temp_RH"), rh_2013.data),
        "u": (("time", "pressure_temp_RH"), u_2013.data),
    },
    coords={
        "time": time_2013,
        "pressure_temp_RH": pressure_temp_RH_2013,
    }
)

ds_2013.to_netcdf("data/2013_temp_RH_u.nc", engine = "netcdf4")

(21, 37)


In [92]:
ds_co2_2013 = xr.open_dataset(path+"2013_CO2/2013_CO2.nc", engine = "netcdf4")
ds_co2_2013

<xarray.Dataset> Size: 41MB
Dimensions:         (valid_time: 21, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 168B 2013-01-07 ... 2013-01-27
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    co2             (valid_time, pressure_level, latitude, longitude) float32 41MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T14:48 GRIB to CDM+CF via cfgrib-0.9.1...

In [93]:
co2_2013 = ds_co2_2013.co2
pressure_chemicals_2013 = ds_co2_2013.pressure_level

# mean over area
co2_2013 = np.mean(co2_2013, axis=3)
co2_2013 = np.mean(co2_2013, axis=2)
print(co2_2013.shape)
co2_2013 = interp_pressure(co2_2013, pressure_chemicals_2013, pressure_temp_RH_2013)

(21, 25)


In [94]:
ds_nox_o3_2013 = xr.open_dataset(path+"2013_NOX_O3/2013_NOX_O3.nc", engine = "netcdf4")
ds_nox_o3_2013

<xarray.Dataset> Size: 124MB
Dimensions:         (valid_time: 21, pressure_level: 25, latitude: 41,
                     longitude: 480)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 168B 2013-01-07 ... 2013-01-27
  * pressure_level  (pressure_level) float64 200B 1e+03 950.0 925.0 ... 2.0 1.0
  * latitude        (latitude) float64 328B 90.0 89.25 88.5 ... 61.5 60.75 60.0
  * longitude       (longitude) float64 4kB -180.0 -179.2 -178.5 ... 178.5 179.2
Data variables:
    no2             (valid_time, pressure_level, latitude, longitude) float32 41MB ...
    no              (valid_time, pressure_level, latitude, longitude) float32 41MB ...
    go3             (valid_time, pressure_level, latitude, longitude) float32 41MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-15T15:06 GRIB to CDM+CF via cfgrib-0.9.1...

In [95]:
no2_2013 = ds_nox_o3_2013.no2
no_2013 = ds_nox_o3_2013.no
o3_2013 = ds_nox_o3_2013.go3

# mean over area:
no2_2013 = np.mean(no2_2013, axis = 3)
no2_2013 = np.mean(no2_2013, axis = 2)
no2_2013 = interp_pressure(no2_2013, pressure_chemicals_2013, pressure_temp_RH_2013)

no_2013 = np.mean(no_2013, axis = 3)
no_2013 = np.mean(no_2013, axis = 2)
no_2013 = interp_pressure(no_2013, pressure_chemicals_2013, pressure_temp_RH_2013)

o3_2013 = np.mean(o3_2013, axis = 3)
o3_2013 = np.mean(o3_2013, axis = 2)
o3_2013 = interp_pressure(o3_2013, pressure_chemicals_2013, pressure_temp_RH_2013)

In [96]:
# create new nc-file:
data_2013 = xr.Dataset(
    data_vars={
        "co2": (("time", "pressure_temp_RH"), co2_2013.data),
        "no2": (("time", "pressure_temp_RH"), no2_2013.data),
        "no": (("time", "pressure_temp_RH"), no_2013.data),
        "o3": (("time", "pressure_temp_RH"), o3_2013.data),
        "t" : (("time", "pressure_temp_RH"), temp_2013.data),
        "u": (("time", "pressure_temp_RH"), u_2013.data),
        "rh": (("time", "pressure_temp_RH"), rh_2013.data),
    },
    coords={
        "time": time_2013,
        "pressure": pressure_temp_RH_2013,
    },
)

data_2013.to_netcdf("data/2013_data.nc", engine = "netcdf4")